In [1]:
import pandas as pd
import numpy as np

TARGETS = ["DeltaTheta", "DeltaX", "DeltaY"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
        Target Dataset  Best_Model  Neurons                Best_R2
0   DeltaTheta     Val  model_12_8       12   [0.8475652932161177]
1       DeltaX     Val  model_14_6       14   [-8.354503962760091]
2       DeltaY     Val  model_16_7       16  [0.30294385863495454]
3   DeltaTheta  Test_1  model_12_5       12   [0.8008670975320218]
4       DeltaX  Test_1  model_14_8       14   [-6.864297470669056]
5       DeltaY  Test_1  model_14_5       14  [-1.1160005895457616]
6   DeltaTheta  Test_2  model_12_6       12   [0.8667435890841706]
7       DeltaX  Test_2  model_14_8       14  [-6.2630175585096595]
8       DeltaY  Test_2  model_16_2       16   [0.8617120234295705]
9   DeltaTheta  Test_3  model_14_8       14   [0.9413390957165708]
10      DeltaX  Test_3  model_16_0       16   [-9.442734375680429]
11      DeltaY  Test_3  model_12_1       12  [0.37229926710024686]
